# Notebook 20: Transfer Learning & Fine-Tuning Pre-trained Models
### Part 20/30 – ML Mastery Series for Python Experts

## Transfer Learning – Why Reinvent the Wheel?

- **ImageNet learned universal visual features**: Models trained on 14M+ images have learned hierarchical representations (edges → textures → patterns → objects) that generalize across domains
- **Reuse lower layers for edges/textures**: Early convolutional layers detect universal features like Gabor filters, color blobs, and simple shapes — these never change
- **Fine-tune higher layers for domain-specific patterns**: Later layers encode domain-specific concepts; these need adaptation to your specific task
- **10×–100× less data & time needed**: Instead of millions of images, you can achieve SOTA with hundreds or thousands
- **Huge boost on small datasets**: Pre-trained weights act as a strong prior, preventing overfitting on limited data
- **Risk of negative transfer if domains too different**: Medical imaging or satellite data may not benefit from natural image pre-training
- **Careful unfreezing strategy required**: Aggressive fine-tuning can destroy pre-trained representations (catastrophic forgetting)
- **Compute efficiency**: Training only the classifier head takes minutes instead of days on GPUs

## Learning Objectives

By the end of this notebook, you will be able to:

- **Load & use pre-trained models** from `keras.applications` (MobileNetV2, ResNet50, EfficientNet)
- **Freeze/unfreeze layers strategically** to preserve pre-trained knowledge while adapting to new domains
- **Build classifier heads** using `GlobalAveragePooling2D` instead of flattening to reduce parameters
- **Apply aggressive data augmentation** to maximize limited training data effectiveness
- **Use learning rate schedulers & warm-up** to stabilize fine-tuning and prevent weight destruction
- **Compare feature extraction vs full fine-tuning** and know when to use each approach
- **Evaluate with confusion matrices & Grad-CAM** for model interpretability and failure analysis
- **Handle small datasets** with techniques like Mixup/Cutout (preview)
- **Recognize when transfer learning fails** and domain adaptation is needed instead
- **Implement production-ready pipelines** for computer vision transfer learning tasks

## 🔄 1. Feature Extraction – Freeze Backbone, Train Only Head

The first phase of transfer learning: we treat the pre-trained CNN as a **fixed feature extractor** and only train a new classification head on top. This is fast, safe, and often achieves 80-90% of the performance with minimal compute.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Configure GPU memory growth to avoid OOM errors
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Load CIFAR-10 dataset and prepare it for transfer learning
# We'll resize to 224x224 since pre-trained models expect larger inputs
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize pixel values to [0, 1] range
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Convert labels to categorical one-hot encoding
num_classes = 10
y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

# Use subset for faster demonstration (10k train, 2k test)
x_train_sub = x_train[:10000]
y_train_sub = y_train_cat[:10000]
x_test_sub = x_test[:2000]
y_test_sub = y_test_cat[:2000]

print(f"Training subset shape: {x_train_sub.shape}")
print(f"Test subset shape: {x_test_sub.shape}")
print(f"Classes: {['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']}")

In [ ]:
# Resize images from 32x32 to 224x224 for pre-trained models
# Using tf.image.resize for better quality than simple upsampling
def resize_images(images, target_size=(224, 224)):
    """Resize batch of images to target size using bilinear interpolation."""
    return tf.image.resize(images, target_size, method='bilinear')

# Preprocess training and test sets
x_train_resized = resize_images(x_train_sub)
x_test_resized = resize_images(x_test_sub)

print(f"Resized training shape: {x_train_resized.shape}")
print(f"Resized test shape: {x_test_resized.shape}")

In [ ]:
# Build feature extraction model with frozen backbone
def build_feature_extraction_model(input_shape=(224, 224, 3), num_classes=10):
    """
    Create transfer learning model with frozen pre-trained backbone.
    Uses MobileNetV2 as feature extractor with custom classification head.
    """
    # Load pre-trained MobileNetV2 without top classification layer
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,  # Exclude ImageNet classification head
        weights='imagenet'  # Load ImageNet pre-trained weights
    )
    
    # Freeze the entire base model - we won't update these weights
    base_model.trainable = False
    
    # Build the complete model
    inputs = keras.Input(shape=input_shape)
    
    # Pass through frozen base model
    x = base_model(inputs, training=False)  # training=False keeps BatchNorm frozen
    
    # Global Average Pooling reduces spatial dimensions without dense parameters
    # Replaces Flatten() which would create too many parameters
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dense hidden layer with ReLU activation
    x = layers.Dense(256, activation='relu')(x)
    
    # Dropout for regularization - prevents overfitting on small dataset
    x = layers.Dropout(0.5)(x)
    
    # Output layer with softmax for multi-class classification
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    return model, base_model

# Create the model
model_fe, base_fe = build_feature_extraction_model()
model_fe.summary()

In [ ]:
# Compile model with standard settings for feature extraction phase
model_fe.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),  # Standard LR for head training
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')]
)

# Verify trainable parameters (should be ~200K, not 3M+)
trainable_count = sum([keras.backend.count_params(w) for w in model_fe.trainable_weights])
non_trainable_count = sum([keras.backend.count_params(w) for w in model_fe.non_trainable_weights])
print(f"Trainable params: {trainable_count:,}")
print(f"Non-trainable params: {non_trainable_count:,}")

In [ ]:
# Train only the classification head (feature extraction phase)
history_fe = model_fe.fit(
    x_train_resized, y_train_sub,
    batch_size=32,
    epochs=15,  # Quick convergence since backbone is frozen
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=3, restore_best_weights=True
        )
    ],
    verbose=1
)

# Evaluate on test set
test_loss, test_acc, test_top3 = model_fe.evaluate(x_test_resized, y_test_sub, verbose=0)
print(f"\nFeature Extraction Results:")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")

## 🎨 2. Data Augmentation for Small Datasets

When you have limited data, augmentation is your best friend. It artificially expands the dataset by applying random transformations that preserve label semantics. For transfer learning, aggressive augmentation helps the model generalize better without destroying pre-trained features.

In [ ]:
# Configure aggressive data augmentation for small dataset training
train_datagen = ImageDataGenerator(
    rotation_range=20,        # Random rotation up to 20 degrees
    width_shift_range=0.2,    # Horizontal shift up to 20% of width
    height_shift_range=0.2,   # Vertical shift up to 20% of height
    shear_range=0.2,          # Shear transformation
    zoom_range=0.2,           # Random zoom in/out up to 20%
    horizontal_flip=True,     # Random horizontal flip
    brightness_range=[0.8, 1.2],  # Random brightness adjustment
    fill_mode='nearest'       # Fill strategy for empty pixels after transform
)

# Validation/test generator - only rescaling, no augmentation
val_datagen = ImageDataGenerator()

# Create generators
train_generator = train_datagen.flow(
    x_train_resized.numpy(), y_train_sub, batch_size=32, seed=42
)
val_generator = val_datagen.flow(
    x_test_resized.numpy(), y_test_sub, batch_size=32, seed=42
)

In [ ]:
# Visualize augmented images to verify augmentation quality
def plot_augmented_images(generator, class_names, n_rows=3, n_cols=3):
    """Display grid of augmented images from generator."""
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 10))
    axes = axes.flatten()
    
    # Get one batch of augmented images
    x_batch, y_batch = next(iter(generator))
    
    for i in range(min(n_rows * n_cols, len(x_batch))):
        ax = axes[i]
        ax.imshow(np.clip(x_batch[i], 0, 1))
        # Get class label from one-hot encoding
        label_idx = np.argmax(y_batch[i])
        ax.set_title(class_names[label_idx], fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.suptitle('Augmented Training Images', fontsize=14, y=1.02)
    plt.show()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']
plot_augmented_images(train_generator, class_names)

In [ ]:
# Advanced augmentation: Random Erasing implementation (simplified version)
def random_erasing(img, probability=0.5, sl=0.02, sh=0.4, r1=0.3):
    """
    Randomly erase a rectangular region of the image.
    Helps reduce overfitting by forcing model to use multiple features.
    """
    if np.random.random() > probability:
        return img
    
    img_h, img_w, img_c = img.shape
    area = img_h * img_w
    
    # Random erase area
    target_area = np.random.uniform(sl, sh) * area
    aspect_ratio = np.random.uniform(r1, 1/r1)
    
    h = int(np.sqrt(target_area * aspect_ratio))
    w = int(np.sqrt(target_area / aspect_ratio))
    
    if h < img_h and w < img_w:
        x1 = np.random.randint(0, img_h - h)
        y1 = np.random.randint(0, img_w - w)
        # Fill with random noise
        img[x1:x1+h, y1:y1+w, :] = np.random.random((h, w, img_c))
    
    return img

# Demonstrate random erasing on one image
sample_img = x_train_resized[0].numpy()
erased_img = random_erasing(sample_img.copy(), probability=1.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(np.clip(sample_img, 0, 1))
ax1.set_title('Original')
ax1.axis('off')
ax2.imshow(np.clip(erased_img, 0, 1))
ax2.set_title('Random Erasing')
ax2.axis('off')
plt.show()

## 🔓 3. Fine-Tuning – Unfreeze Layers Gradually

After training the head, we unfreeze select layers of the backbone for fine-tuning. **Critical rules**: (1) Use very low learning rates (1e-5 to 1e-4), (2) Unfreeze gradually from top-down, (3) Keep BatchNorm frozen unless you have large batch sizes. This adapts high-level features while preserving low-level ones.

In [ ]:
# Prepare model for fine-tuning by unfreezing top layers
def setup_fine_tuning(model, base_model, unfreeze_from_layer=100):
    """
    Unfreeze layers from specified index onwards for fine-tuning.
    Lower layers (earlier indices) stay frozen to preserve edge detectors.
    """
    # First, enable training for the base model
    base_model.trainable = True
    
    # Freeze all layers before the cutoff
    for layer in base_model.layers[:unfreeze_from_layer]:
        layer.trainable = False
    
    # Count trainable vs frozen in backbone
    trainable_backbone = sum(1 for l in base_model.layers if l.trainable)
    total_backbone = len(base_model.layers)
    
    print(f"Backbone layers: {trainable_backbone}/{total_backbone} trainable")
    print(f"First trainable layer: {base_model.layers[unfreeze_from_layer].name}")
    
    return model, base_model

# Unfreeze from layer 100 (MobileNetV2 has ~150 layers total)
model_ft, base_ft = setup_fine_tuning(model_fe, base_fe, unfreeze_from_layer=100)

In [ ]:
# Recompile with MUCH lower learning rate for fine-tuning
# This prevents destroying pre-trained weights
model_ft.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # 100x smaller than before!
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')]
)

# Verify parameter counts changed
trainable_count = sum([keras.backend.count_params(w) for w in model_ft.trainable_weights])
non_trainable_count = sum([keras.backend.count_params(w) for w in model_ft.non_trainable_weights])
print(f"\nAfter unfreezing:")
print(f"Trainable params: {trainable_count:,}")
print(f"Non-trainable params: {non_trainable_count:,}")

In [ ]:
# Fine-tune with callbacks for learning rate reduction and early stopping
callbacks_ft = [
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    )
)

# Fine-tuning training
history_ft = model_ft.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=callbacks_ft,
    verbose=1
)

# Final evaluation
test_loss_ft, test_acc_ft, test_top3_ft = model_ft.evaluate(x_test_resized, y_test_sub, verbose=0)
print(f"\n{'='*50}")
print(f"FINAL RESULTS COMPARISON:")
print(f"{'='*50}")
print(f"Feature Extraction Only:")
print(f"  Test Accuracy: {test_acc:.4f} | Top-3: {test_top3:.4f}")
print(f"Fine-Tuned:")
print(f"  Test Accuracy: {test_acc_ft:.4f} | Top-3: {test_top3_ft:.4f}")
print(f"Improvement: +{test_acc_ft - test_acc:.4f} ({((test_acc_ft/test_acc - 1)*100):.1f}%)")

In [ ]:
# Plot training curves comparison
def plot_training_comparison(history_fe, history_ft):
    """Compare feature extraction vs fine-tuning training curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy plot
    ax1 = axes[0]
    ax1.plot(history_fe.history['accuracy'], label='FE Train', linestyle='--')
    ax1.plot(history_fe.history['val_accuracy'], label='FE Val', linestyle='--')
    ax1.plot(history_ft.history['accuracy'], label='FT Train', linewidth=2)
    ax1.plot(history_ft.history['val_accuracy'], label='FT Val', linewidth=2)
    ax1.set_title('Model Accuracy Comparison')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Loss plot
    ax2 = axes[1]
    ax2.plot(history_fe.history['loss'], label='FE Train', linestyle='--')
    ax2.plot(history_fe.history['val_loss'], label='FE Val', linestyle='--')
    ax2.plot(history_ft.history['loss'], label='FT Train', linewidth=2)
    ax2.plot(history_ft.history['val_loss'], label='FT Val', linewidth=2)
    ax2.set_title('Model Loss Comparison')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_training_comparison(history_fe, history_ft)

## 🏗️ 4. Comparing Different Backbones

Different architectures offer different trade-offs between speed, size, and accuracy. Let's compare three popular choices on the same dataset to understand these trade-offs empirically.

In [ ]:
from tensorflow.keras.applications import ResNet50V2, EfficientNetB0
import time

def build_and_train_backbone(backbone_name, x_train, y_train, x_val, y_val, epochs=10):
    """
    Build, train, and evaluate a transfer learning model with specified backbone.
    Returns metrics dictionary for comparison.
    """
    print(f"\n{'='*60}")
    print(f"Training with backbone: {backbone_name}")
    print(f"{'='*60}")
    
    # Select backbone architecture
    if backbone_name == 'MobileNetV2':
        base = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
    elif backbone_name == 'ResNet50V2':
        base = ResNet50V2(input_shape=(224,224,3), include_top=False, weights='imagenet')
    elif backbone_name == 'EfficientNetB0':
        base = EfficientNetB0(input_shape=(224,224,3), include_top=False, weights='imagenet')
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")
    
    # Freeze backbone
    base.trainable = False
    
    # Build model
    inputs = keras.Input(shape=(224,224,3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    
    # Count parameters
    trainable_params = sum([keras.backend.count_params(w) for w in model.trainable_weights])
    total_params = model.count_params()
    
    # Compile
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Train and time it
    start_time = time.time()
    history = model.fit(
        x_train, y_train,
        batch_size=32,
        epochs=epochs,
        validation_data=(x_val, y_val),
        verbose=0
    )
    train_time = time.time() - start_time
    
    # Evaluate
    val_loss, val_acc = model.evaluate(x_val, y_val, verbose=0)
    
    return {
        'backbone': backbone_name,
        'total_params': total_params,
        'trainable_params': trainable_params,
        'training_time': train_time,
        'final_val_acc': val_acc,
        'final_val_loss': val_loss,
        'history': history
    }

# Run comparison (using smaller subset for speed)
x_val_sub = x_test_resized[:1000]
y_val_sub = y_test_sub[:1000]

results = []
for backbone in ['MobileNetV2', 'ResNet50V2', 'EfficientNetB0']:
    result = build_and_train_backbone(
        backbone, x_train_resized[:5000], y_train_sub[:5000], 
        x_val_sub, y_val_sub, epochs=8
    )
    results.append(result)

In [ ]:
# Display comparison table
import pandas as pd

comparison_df = pd.DataFrame([
    {
        'Model': r['backbone'],
        'Total Params (M)': f"{r['total_params']/1e6:.2f}",
        'Trainable Params (K)': f"{r['trainable_params']/1e3:.1f}",
        'Training Time (s)': f"{r['training_time']:.1f}",
        'Val Accuracy': f"{r['final_val_acc']:.4f}"
    }
    for r in results
])

print("\n" + "="*80)
print("BACKBONE COMPARISON TABLE")
print("="*80)
print(comparison_df.to_string(index=False))
print("\nKey Insights:")
print("- MobileNetV2: Fastest, smallest, good for mobile/edge deployment")
print("- ResNet50V2: Balanced performance, well-established architecture")  
print("- EfficientNetB0: Best accuracy/parameter ratio, compound scaling")

## 📉 5. Learning Rate Scheduling & Warm-up

Learning rate is the most critical hyperparameter in fine-tuning. Too high = destroyed pre-trained weights. Too low = slow convergence. Schedulers adapt LR during training, and warm-up prevents early instability.

In [ ]:
# Implement cosine decay scheduler with warm-up
class CosineDecayWithWarmup(keras.callbacks.Callback):
    """
    Cosine annealing learning rate schedule with linear warm-up phase.
    Warm-up prevents early training instability in fine-tuning.
    """
    def __init__(self, target_lr, warmup_epochs, total_epochs):
        super().__init__()
        self.target_lr = target_lr
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.lrs = []
    
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            # Linear warm-up
            lr = self.target_lr * (epoch + 1) / self.warmup_epochs
        else:
            # Cosine decay after warm-up
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.target_lr * 0.5 * (1 + np.cos(np.pi * progress))
        
        self.model.optimizer.learning_rate.assign(lr)
        self.lrs.append(lr)
        print(f"\nEpoch {epoch+1}: LR = {lr:.2e}")

# Alternative: Exponential decay using built-in scheduler
def exponential_decay_scheduler(epoch, lr):
    """Reduce LR by 5% every epoch after epoch 10."""
    if epoch < 10:
        return lr
    return lr * 0.95

exp_scheduler = keras.callbacks.LearningRateScheduler(exponential_decay_scheduler)

In [ ]:
# Demonstrate LR scheduling with a fresh model
model_sched = build_feature_extraction_model()[0]
model_sched.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-6),  # Start low
                    loss='categorical_crossentropy', metrics=['accuracy'])

# Setup cosine decay with 5-epoch warm-up
lr_scheduler = CosineDecayWithWarmup(
    target_lr=1e-4, warmup_epochs=5, total_epochs=20
)

# Train with scheduler
history_sched = model_sched.fit(
    x_train_resized[:3000], y_train_sub[:3000],
    batch_size=32,
    epochs=20,
    validation_split=0.2,
    callbacks=[lr_scheduler],
    verbose=0
)

# Plot learning rate schedule
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(lr_scheduler.lrs)+1), lr_scheduler.lrs, 'b-o', linewidth=2)
plt.title('Learning Rate Schedule: Warm-up + Cosine Decay')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.axvline(x=5, color='r', linestyle='--', label='Warm-up ends')
plt.legend()
plt.show()

print(f"Initial LR: {lr_scheduler.lrs[0]:.2e}")
print(f"Peak LR: {max(lr_scheduler.lrs):.2e}")
print(f"Final LR: {lr_scheduler.lrs[-1]:.2e}")

## 🔍 6. Interpretability – Grad-CAM on Predictions

Grad-CAM (Gradient-weighted Class Activation Mapping) shows where the model "looks" to make predictions. Essential for debugging domain shift and verifying the model uses relevant features, not spurious correlations.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Generate Grad-CAM heatmap for input image.
    Uses gradients of target class flowing into final conv layer.
    """
    # Create model that maps input to activations of last conv layer and output
    grad_model = keras.models.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    # Compute gradient of top predicted class for input image
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        target_class_output = predictions[:, pred_index]
    
    # Gradient of output neuron w.r.t. conv layer output
    grads = tape.gradient(target_class_output, conv_outputs)
    
    # Global average pooling of gradients (weights for each feature map)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Weight feature maps by importance, sum to get heatmap
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    
    # ReLU to focus on features that positively influence the class
    heatmap = tf.maximum(heatmap, 0)
    # Normalize to [0, 1]
    heatmap = heatmap / tf.reduce_max(heatmap)
    
    return heatmap.numpy()

def superimpose_heatmap(img, heatmap, alpha=0.4):
    """Overlay heatmap on original image."""
    # Resize heatmap to match image
    heatmap = tf.image.resize(heatmap[..., np.newaxis], [img.shape[0], img.shape[1]])
    heatmap = heatmap.numpy().squeeze()
    
    # Convert to RGB heatmap (jet colormap)
    heatmap = np.uint8(255 * heatmap)
    jet = plt.cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    
    # Superimpose
    superimposed = jet_heatmap * alpha + img * (1 - alpha)
    return np.clip(superimposed, 0, 1)

In [ ]:
# Find last conv layer name in MobileNetV2
for layer in model_ft.layers:
    if isinstance(layer, keras.Model):  # Base model
        for sublayer in layer.layers:
            if isinstance(sublayer, layers.Conv2D):
                last_conv = sublayer.name
        print(f"Last conv layer in backbone: {last_conv}")

# Get correct and incorrect predictions
predictions = model_ft.predict(x_test_resized[:100], verbose=0)
pred_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test_sub[:100], axis=1)

correct_idx = np.where(pred_classes == true_classes)[0]
incorrect_idx = np.where(pred_classes != true_classes)[0]

print(f"Correct predictions: {len(correct_idx)}, Incorrect: {len(incorrect_idx)}")

In [ ]:
# Visualize Grad-CAM for correct and incorrect predictions
def visualize_gradcam_examples(model, x_data, y_true, indices, title, last_conv_name='Conv_1'):
    """Display Grad-CAM visualizations for selected indices."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, idx in enumerate(indices[:3]):
        img = x_data[idx].numpy()
        img_batch = np.expand_dims(img, axis=0)
        
        # Get prediction
        pred = model.predict(img_batch, verbose=0)
        pred_class = np.argmax(pred[0])
        true_class = np.argmax(y_true[idx])
        
        # Generate heatmap
        heatmap = make_gradcam_heatmap(img_batch, model, last_conv_name)
        superimposed = superimpose_heatmap(img, heatmap)
        
        # Plot original
        axes[i*2].imshow(np.clip(img, 0, 1))
        axes[i*2].set_title(f'{title} #{idx}\nTrue: {class_names[true_class]}\nPred: {class_names[pred_class]}')
        axes[i*2].axis('off')
        
        # Plot Grad-CAM
        axes[i*2+1].imshow(superimposed)
        axes[i*2+1].set_title('Grad-CAM Attention')
        axes[i*2+1].axis('off')
    
    plt.tight_layout()
    plt.show()

print("CORRECT PREDICTIONS - Grad-CAM Analysis:")
visualize_gradcam_examples(model_ft, x_test_resized, y_test_sub, correct_idx, "Correct")

print("\nINCORRECT PREDICTIONS - Grad-CAM Analysis:")
visualize_gradcam_examples(model_ft, x_test_resized, y_test_sub, incorrect_idx, "Wrong")

## 🎯 7. Small Custom Dataset Simulation – Transfer Wins Big

The true power of transfer learning shines with extremely small datasets. Let's simulate having only 100-500 images per class and compare training from scratch vs. transfer learning.

In [ ]:
# Simulate very small dataset: 100 samples per class (1000 total)
def create_small_dataset(x, y, samples_per_class=100):
    """Create balanced small dataset with specified samples per class."""
    x_small, y_small = [], []
    y_int = np.argmax(y, axis=1)
    
    for class_id in range(10):
        idx = np.where(y_int == class_id)[0][:samples_per_class]
        x_small.append(x[idx])
        y_small.append(y[idx])
    
    x_small = np.concatenate(x_small)
    y_small = np.concatenate(y_small)
    
    # Shuffle
    perm = np.random.permutation(len(x_small))
    return x_small[perm], y_small[perm]

x_tiny, y_tiny = create_small_dataset(x_train_resized.numpy(), y_train_sub, samples_per_class=100)
print(f"Tiny dataset shape: {x_tiny.shape}, {y_tiny.shape}")

In [ ]:
# Model 1: Train simple CNN from scratch
def build_scratch_cnn(input_shape=(224,224,3), num_classes=10):
    """Simple CNN trained from scratch for comparison."""
    model = keras.Sequential([
        layers.Conv2D(32, 3, activation='relu', input_shape=input_shape),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model_scratch = build_scratch_cnn()
model_scratch.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("Training FROM SCRATCH on tiny dataset...")
history_scratch = model_scratch.fit(
    x_tiny, y_tiny,
    batch_size=16,
    epochs=30,
    validation_split=0.2,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)

scratch_loss, scratch_acc = model_scratch.evaluate(x_test_resized, y_test_sub, verbose=0)

In [ ]:
# Model 2: Transfer learning with same tiny dataset
base_tiny = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base_tiny.trainable = False

inputs = keras.Input(shape=(224,224,3))
x = base_tiny(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)
model_tiny = keras.Model(inputs, outputs)

model_tiny.compile(optimizer=keras.optimizers.Adam(1e-3), 
                   loss='categorical_crossentropy', metrics=['accuracy'])

print("Training with TRANSFER LEARNING on tiny dataset...")
history_tiny = model_tiny.fit(
    x_tiny, y_tiny,
    batch_size=16,
    epochs=30,
    validation_split=0.2,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)

tiny_loss, tiny_acc = model_tiny.evaluate(x_test_resized, y_test_sub, verbose=0)

In [ ]:
# Compare results
print("\n" + "="*60)
print("SMALL DATASET SHOWDOWN: 100 images per class")
print("="*60)
print(f"Train from Scratch:     {scratch_acc:.4f} accuracy")
print(f"Transfer Learning:      {tiny_acc:.4f} accuracy")
print(f"Advantage:              +{tiny_acc - scratch_acc:.4f} ({((tiny_acc/scratch_acc - 1)*100):.1f}% relative)")
print("="*60)

# Visualize comparison
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
methods = ['From Scratch', 'Transfer Learning']
accuracies = [scratch_acc, tiny_acc]
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(methods, accuracies, color=colors, edgecolor='black', linewidth=2)
ax.set_ylim(0, 1)
ax.set_ylabel('Test Accuracy')
ax.set_title('Small Dataset (100 samples/class): Transfer Learning vs Scratch')

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## ⚠️ Common Pitfalls & Pro Tips

- **Unfreezing too many layers too early** → Catastrophic forgetting destroys pre-trained features. Always start with the head only.
- **High learning rate during fine-tuning** → Even 1e-3 can destroy ImageNet weights. Use 1e-5 to 1e-4 maximum.
- **Weak augmentation on small data** → Still overfits. Use aggressive augmentation: rotation, shear, zoom, color jitter.
- **Input size mismatch** → Resizing 32×32 to 224×224 creates artifacts. Use proper upsampling or choose appropriate input size.
- **Not freezing BatchNorm** → During fine-tuning with small batches, BN statistics get corrupted. Use `training=False` or freeze BN layers.
- **Ignoring mixed precision** → `tf.keras.mixed_precision.Policy('mixed_float16')` speeds up training 2-3× on modern GPUs.
- **Not monitoring validation loss** → Training accuracy can increase while validation loss increases (overfitting). Use EarlyStopping.
- **Negative transfer on very different domains** → Medical imaging, satellite, or industrial inspection may need domain adaptation, not just fine-tuning.
- **Forgetting class weight balancing** → Small datasets often have imbalanced classes. Use `class_weight` parameter in `model.fit()`.
- **Not using transfer learning for similar domains** → If your data is similar to ImageNet (natural images), you're leaving accuracy on the table by training from scratch.
- **One-shot fine-tuning** → Better to do iterative unfreezing: head → top 20% → top 50% → full model, with LR adjustments.
- **Not saving best weights** → Always use `ModelCheckpoint` and `EarlyStopping` with `restore_best_weights=True`.

## 📝 Exercises

Test your understanding with these hands-on challenges:

**Easy:** Apply MobileNetV2 feature extraction on Fashion-MNIST resized to 224×224. Compare accuracy to training from scratch on the 28×28 original resolution.

**Medium:** Fine-tune EfficientNetB0 on CIFAR-10 subset with the following constraints: (1) Use cosine decay scheduler, (2) Unfreeze only top 30 layers, (3) Aim for >85% test accuracy. Document your LR choices.

**Medium:** Implement cosine annealing scheduler from scratch (without built-in callbacks) and compare convergence speed to constant learning rate on the same model architecture.

**Hard:** Add Grad-CAM visualization for the 5 most confident misclassified images in the test set. Analyze what the model is looking at and why it might be confused (e.g., background clutter, similar textures).

**Bonus:** Simulate extremely small data regime (50 images per class). Compare: (A) Transfer learning with heavy augmentation, (B) Transfer learning with light augmentation, (C) Training from scratch with heavy augmentation. Present learning curves and final accuracies in a table.

<details>
<summary><b>Exercise Solutions (Click to Expand)</b></summary>

### Easy Solution Outline
```python
# Load Fashion-MNIST
(x_fashion, y_fashion), (x_fashion_test, y_fashion_test) = keras.datasets.fashion_mnist.load_data()
# Add channel dimension, normalize, resize to 224x224
x_fashion = np.stack([x_fashion]*3, axis=-1)  # Grayscale to RGB
# ... rest follows same pattern as CIFAR-10 example
```

### Medium Solution Outline (EfficientNetB0)
```python
base = EfficientNetB0(input_shape=(224,224,3), include_top=False, weights='imagenet')
base.trainable = True
for layer in base.layers[:-30]:  # Freeze all except top 30
    layer.trainable = False
# Use LR = 1e-5 for fine-tuning, cosine decay after warm-up
```

### Hard Solution Outline (Grad-CAM Analysis)
```python
# Find high-confidence wrong predictions
probs = model.predict(x_test)
confidences = np.max(probs, axis=1)
wrong = np.where(pred_classes != true_classes)[0]
top_wrong = wrong[np.argsort(confidences[wrong])[-5:]]
# Apply Grad-CAM to these indices
```

</details>

## ✅ Summary – What You Learned Today

- **Transfer learning fundamentals**: Leverage ImageNet pre-trained features to bootstrap your computer vision models
- **Two-phase training strategy**: Feature extraction (frozen backbone) → Fine-tuning (gradual unfreezing)
- **Architecture trade-offs**: MobileNetV2 for speed/edge, ResNet50 for balance, EfficientNet for accuracy/efficiency
- **Critical hyperparameters**: Learning rate (10-100× smaller in fine-tuning), batch normalization behavior, augmentation strength
- **Advanced techniques**: Learning rate scheduling (warm-up + cosine decay), Grad-CAM for interpretability
- **Small data regimes**: Transfer learning provides massive gains (often 20-40% accuracy improvement) when data is scarce
- **Failure modes**: Negative transfer, catastrophic forgetting, and domain mismatch – and how to detect them

## 🔮 Next Notebook Preview

**Notebook 21: Recurrent Neural Networks, LSTMs & Sequence Modeling**

Moving beyond images into the temporal dimension! We'll explore:
- **Time-series forecasting** and sequential data processing
- **Vanilla RNNs** and why they suffer from vanishing gradients
- **LSTM & GRU gates**: The mechanics of long-term memory
- **Bidirectional RNNs**: Using future context for better predictions
- **Attention mechanisms preview**: The foundation of modern transformers
- Applications in text classification, sentiment analysis, and signal processing

Get ready to give your neural networks memory! 🧠⏳